# Module 9 · 影片 2：影格抽樣與前處理

> **本節定位｜全新（2026 必備）**
> 一段 10 秒、30fps 的影片有 300 張影格，且相鄰影格高度冗餘。
> 影片模型只吃**少量代表影格**（如 8、16、32 張），因此**影格抽樣策略**是關鍵。

## 學習目標
- 掌握三種抽樣策略：**均勻 (uniform)**、**密集 (dense)**、**分段 (segment/TSN)**。
- 用 VideoMAE 的 `AutoImageProcessor` 把影格標準化成模型輸入 `(N, T, C, H, W)`。
- 釐清 **clip-level vs frame-level** 的差異。

In [ ]:
import numpy as np
import av
from huggingface_hub import hf_hub_download

# 真實影片：解碼出全部影格，取得真實影格數
video_path = hf_hub_download(repo_id="nielsr/video-demo",
                             filename="eating_spaghetti.mp4", repo_type="dataset")
container = av.open(video_path)
all_frames = [f.to_ndarray(format="rgb24") for f in container.decode(video=0)]
container.close()
total_frames = len(all_frames)
print(f"真實影片 eating_spaghetti.mp4 共 {total_frames} 影格；目標抽樣 num_frames=16 張")

## 1. 三種抽樣策略

In [ ]:
num_frames = 16

# (a) 均勻抽樣：整段等距取樣（最常用，能涵蓋全片）
uniform = np.linspace(0, total_frames - 1, num_frames).astype(int)

# (b) 密集抽樣：從某起點連續取（適合需要細緻動作的片段）
start = max(0, min(20, total_frames - num_frames))
dense = np.clip(np.arange(start, start + num_frames), 0, total_frames - 1)

# (c) 分段抽樣 (TSN)：切成 num_frames 段，每段隨機取一張（訓練時增加多樣性）
seg_bounds = np.linspace(0, total_frames, num_frames + 1).astype(int)
rs = np.random.RandomState(0)
segment = np.array([rs.randint(seg_bounds[i], max(seg_bounds[i]+1, seg_bounds[i+1]))
                    for i in range(num_frames)])

print("均勻抽樣索引:", uniform)
print("密集抽樣索引:", dense)
print("分段抽樣索引:", segment)
print("\n→ 均勻：涵蓋全片；密集：聚焦局部；分段：訓練時的隨機增強。")

## 2. VideoMAE Processor：標準化成模型輸入

和圖像的 `AutoImageProcessor` 一樣，影片模型有配套 processor，自動 resize/normalize
並輸出 `pixel_values (N, T, C, H, W)`。需 `multimodal` extra。

In [ ]:
# 取真實影片在「均勻抽樣」索引上的 16 張影格
sampled = [all_frames[i] for i in uniform]

try:
    from transformers import AutoImageProcessor
    proc = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
    out = proc(list(sampled), return_tensors="pt")
    print(f"VideoMAE pixel_values: {tuple(out['pixel_values'].shape)}  (N, T, C, H, W)")
    print("→ 一段 clip 的 16 張影格被整理成模型期望的 5 維張量。")
except Exception as e:
    print("（未能下載 VideoMAE processor，略過）", e)
    print("概念：processor 會把抽樣影格 resize/normalize 成 (N, T, C, H, W)。")

## 3. Clip-level vs Frame-level

| 策略 | 作法 | 適用 |
|:--|:--|:--|
| **Frame-level** | 每張影格獨立過影像模型，再聚合 | 動作不強依賴時間順序 |
| **Clip-level** | 整段影格一起進影片模型（學時間關係） | 動作辨識、需時序的任務 |

2026 的影片 Transformer（VideoMAE/ViViT）多採 **clip-level**，因為它們能用
**時空注意力**同時建模空間與時間。

## 小結

| 重點 | 內容 |
|:--|:--|
| 為何抽樣 | 影片長且冗餘，模型只吃少量代表影格 |
| 三策略 | 均勻（涵蓋全片）、密集（聚焦局部）、分段（訓練增強） |
| 標準化 | VideoMAE processor → `(N, T, C, H, W)` |
| clip vs frame | 現代影片 Transformer 多採 clip-level 時空建模 |

**下一節 `03_video_case`**：用預訓練 VideoMAE 對一段 clip 做動作辨識推論。